# Google Play Store Market Analysis

**Objective:** Clean app metadata, analyse categories/ratings/installs/pricing and classify review sentiment.

> Dataset note: included CSVs are synthetic practice data created for this portfolio. Replace with approved public datasets for official internship submission.


## Setup
Run the following cell. Install missing packages with `pip install -r ../../requirements.txt` from the repository root.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
apps=pd.read_csv('play_store_apps_synthetic.csv'); reviews=pd.read_csv('play_store_reviews_synthetic.csv')
apps=apps.drop_duplicates(subset='App').copy(); apps.Rating=apps.Rating.fillna(apps.Rating.median())
apps['Revenue_proxy']=np.where(apps.Type=='Paid',apps.Price*apps.Installs,0)
apps.Category.value_counts().plot(kind='bar',title='App count by category'); plt.show()
apps.groupby('Category').Rating.mean().sort_values().plot(kind='barh',title='Average rating by category'); plt.show()
sns.scatterplot(data=apps,x='Size_MB',y='Installs',hue='Type'); plt.xscale('log'); plt.yscale('log'); plt.show()
apps.Type.value_counts().plot(kind='pie',autopct='%1.1f%%'); plt.ylabel(''); plt.title('Free vs paid'); plt.show()
display(apps[apps.Type=='Paid'].Price.describe())
display(apps.groupby('Category').Revenue_proxy.sum().sort_values(ascending=False))
Xtr,Xte,ytr,yte=train_test_split(reviews.Review,reviews.Sentiment,test_size=.2,random_state=42,stratify=reviews.Sentiment)
vec=TfidfVectorizer(stop_words='english'); A=vec.fit_transform(Xtr); B=vec.transform(Xte)
model=LogisticRegression(max_iter=1000).fit(A,ytr); pred=model.predict(B)
print(classification_report(yte,pred))
reviews['Predicted_Sentiment']=model.predict(vec.transform(reviews.Review))
display(pd.crosstab(reviews.Category,reviews.Predicted_Sentiment,normalize='index').round(2))
print('Developer insights: category saturation can guide differentiation; ratings and review sentiment highlight quality gaps; price and install proxies help evaluate monetisation.')
